# 05b - Game Predictions (Binary Ensemble)

Predict full game statlines for starting pitchers using the binary model ensemble.

**Key differences from 05:**
- Uses 7 separate binary classifiers instead of one 7-class model
- Each outcome model has its own optimized features
- Should have better calibration for elite pitchers

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from datetime import date, timedelta
from pathlib import Path

from src.game_predictor_binary import GamePredictorBinary, format_prediction_summary
from src.data.mlb_api import get_games_with_lineups, check_lineup_status

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

/Applications/miniconda3/envs/mlbenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Memory management utilities
import gc
import psutil

def clear_mem():
    """Run garbage collection and print memory usage."""
    gc.collect()
    mem_gb = psutil.Process().memory_info().rss / 1024**3
    print(f"Memory: {mem_gb:.2f} GB")
    return mem_gb

clear_mem()

Memory: 0.30 GB


0.2995109558105469

## 1. Initialize Predictor

In [3]:
# Initialize predictor with binary ensemble
predictor = GamePredictorBinary(
    ensemble_dir='../models/binary_ensemble',
    preprocessor_path='../models/matchup_preprocessor.pkl',
    pitcher_profiles_path='../data/profiles/pitcher_arsenal.csv',
    batter_profiles_path='../data/profiles/batter_profiles.csv',
    pitcher_rolling_path='../data/profiles/pitcher_rolling.csv',
    batter_rolling_path='../data/profiles/batter_rolling.csv',
    park_factors_path= "../data/profiles/park_factors.csv"
)

print(f"Feature names: {len(predictor.feature_names)}")
print(f"Outcome classes: {predictor.outcome_classes}")

Feature names: 207
Outcome classes: ['1B', '2B', '3B', 'BB', 'HR', 'K', 'OUT']


## 2. Check Today's Games

In [4]:
# Check lineup availability
today = date.today().strftime("%Y-%m-%d")
print(f"Checking games for {today}...")

status = check_lineup_status(today)
print(f"\nTotal games: {status['total_games']}")
print(f"With lineups: {status['games_with_lineups']}")
print(f"With probable pitchers: {status['games_with_probable_pitchers']}")

print("\nGames:")
for game in status["games"]:
    away = game["away_team"]["abbreviation"]
    home = game["home_team"]["abbreviation"]
    away_p = game["away_pitcher"]["pitcher_name"] if game["away_pitcher"] else "TBD"
    home_p = game["home_pitcher"]["pitcher_name"] if game["home_pitcher"] else "TBD"
    lineups = "Lineups" if game["lineups_available"] else "No lineups"
    print(f"  {away} @ {home}: {away_p} vs {home_p} [{lineups}]")

Checking games for 2026-04-10...

Total games: 15
With lineups: 15
With probable pitchers: 15

Games:
  PIT @ CHC: Carmen Mlodzinski vs Shota Imanaga [Lineups]
  AZ @ PHI: Michael Soroka vs Jesús Luzardo [Lineups]
  MIA @ DET: Chris Paddack vs Keider Montero [Lineups]
  LAA @ CIN: Jack Kochanowicz vs Chase Burns [Lineups]
  MIN @ TOR: Simeon Woods Richardson vs Patrick Corbin [Lineups]
  ATH @ NYM: J.T. Ginn vs Clay Holmes [Lineups]
  NYY @ TB: Luis Gil vs Steven Matz [Lineups]
  SF @ BAL: Landen Roupp vs Shane Baz [Lineups]
  CLE @ ATL: Slade Cecconi vs Bryce Elder [Lineups]
  CWS @ KC: Davis Martin vs Kris Bubic [Lineups]
  WSH @ MIL: Jake Irvin vs Aaron Ashby [Lineups]
  BOS @ STL: Connelly Early vs Dustin May [Lineups]
  COL @ SD: Tomoyuki Sugano vs Walker Buehler [Lineups]
  HOU @ SEA: Tatsuya Imai vs Emerson Hancock [Lineups]
  TEX @ LAD: Kumar Rocker vs Tyler Glasnow [Lineups]


## 3. Run Predictions

In [5]:
# Predict all games for today
predictions = predictor.predict_day(today)

print(f"\n{'='*60}")
print(f"PREDICTIONS FOR {today}")
print(f"{'='*60}")

for game in predictions:
    print(f"\n{game['away_team']} @ {game['home_team']} ({game['status']})")
    print("-" * 40)
    
    if game["away_prediction"]:
        print(format_prediction_summary(game["away_prediction"]))
    
    if game["home_prediction"]:
        print(format_prediction_summary(game["home_prediction"]))
    
    if game["errors"]:
        print(f"  Errors: {game['errors']}")


PREDICTIONS FOR 2026-04-10

PIT @ CHC (complete)
----------------------------------------
Carmen Mlodzinski
  Target IP: 4.4, Sim IP: 4.3
  K: 4.0, BB: 1.9, H: 4.5, HR: 0.5
  Expected Runs: 2.20
  ERA (this start): 4.57
Shota Imanaga
  Target IP: 5.4, Sim IP: 5.3
  K: 5.9, BB: 1.8, H: 5.2, HR: 0.8
  Expected Runs: 2.41
  ERA (this start): 4.07

AZ @ PHI (complete)
----------------------------------------
Michael Soroka
  Target IP: 4.3, Sim IP: 4.3
  K: 4.1, BB: 1.7, H: 4.4, HR: 0.6
  Expected Runs: 1.90
  ERA (this start): 3.94
Jesús Luzardo
  Target IP: 6.1, Sim IP: 6.0
  K: 6.3, BB: 2.0, H: 5.3, HR: 0.8
  Expected Runs: 2.19
  ERA (this start): 3.28

MIA @ DET (complete)
----------------------------------------
Chris Paddack
  Target IP: 4.4, Sim IP: 4.3
  K: 3.9, BB: 1.8, H: 4.4, HR: 0.6
  Expected Runs: 2.08
  ERA (this start): 4.33
Keider Montero
  Target IP: 4.6, Sim IP: 4.7
  K: 3.8, BB: 2.0, H: 5.2, HR: 0.7
  Expected Runs: 2.46
  ERA (this start): 4.75

LAA @ CIN (complete)


## 4. Detailed View - Single Game

In [6]:
# Show detailed predictions for first complete game
complete_games = [g for g in predictions if g['status'] == 'complete']

if complete_games:
    game = complete_games[0]
    print(f"Detailed view: {game['away_team']} @ {game['home_team']}")
    print("=" * 60)
    
    for side, pred in [("AWAY", game["away_prediction"]), ("HOME", game["home_prediction"])]:
        if pred:
            print(f"\n{side}: {pred['pitcher_name']}")
            print(f"Expected BF: {pred['expected_bf']}")
            print(f"\nExpected Stats:")
            for stat, val in pred['expected_stats'].items():
                print(f"  {stat}: {val}")
            
            print(f"\nPer-Batter Predictions (first time through):")
            for bp in pred['batter_predictions'][:9]:
                k_prob = bp['probabilities']['K']
                bb_prob = bp['probabilities']['BB']
                hr_prob = bp['probabilities']['HR']
                print(f"  {bp['batter_name'][:20]:20s}: K={k_prob:.1%}, BB={bb_prob:.1%}, HR={hr_prob:.1%}")
else:
    print("No complete games available")

Detailed view: PIT @ CHC

AWAY: Carmen Mlodzinski
Expected BF: 21.0

Expected Stats:
  K: 3.95
  BB: 1.9
  H: 4.55
  1B: 2.92
  2B: 1.06
  3B: 0.05
  HR: 0.52
  ER: 2.2
  IP_approx: 4.3
  ERA_approx: 4.57

Per-Batter Predictions (first time through):
  Nico Hoerner        : K=11.9%, BB=9.5%, HR=3.4%
  Michael Busch       : K=21.0%, BB=8.7%, HR=3.4%
  Alex Bregman        : K=14.5%, BB=9.8%, HR=3.7%
  Ian Happ            : K=24.7%, BB=8.9%, HR=3.1%
  Seiya Suzuki        : K=24.4%, BB=8.8%, HR=3.3%
  Pete Crow-Armstrong : K=23.5%, BB=8.1%, HR=3.2%
  Carson Kelly        : K=20.9%, BB=8.7%, HR=3.3%
  Moisés Ballesteros  : K=19.7%, BB=9.2%, HR=3.3%
  Dansby Swanson      : K=26.3%, BB=8.1%, HR=3.0%

HOME: Shota Imanaga
Expected BF: 22.0

Expected Stats:
  K: 5.89
  BB: 1.76
  H: 5.19
  1B: 3.19
  2B: 1.1
  3B: 0.08
  HR: 0.82
  ER: 2.41
  IP_approx: 5.3
  ERA_approx: 4.07

Per-Batter Predictions (first time through):
  Nick Yorke          : K=18.3%, BB=9.0%, HR=3.2%
  Ryan O'Hearn        : K=

## 5. Compare with Yesterday (Completed Games)

In [7]:
# Get yesterday's games (should have lineups from boxscore)
yesterday = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")
print(f"Predictions for yesterday ({yesterday}):")

yesterday_preds = predictor.predict_day(yesterday)

for game in yesterday_preds:
    if game['status'] == 'complete':
        print(f"\n{game['away_team']} @ {game['home_team']}")
        
        if game["away_prediction"]:
            p = game["away_prediction"]
            s = p["expected_stats"]
            print(f"  {p['pitcher_name']}: {s['K']:.1f} K, {s['BB']:.1f} BB, {s['H']:.1f} H, ERA={s['ERA_approx']}")
        
        if game["home_prediction"]:
            p = game["home_prediction"]
            s = p["expected_stats"]
            print(f"  {p['pitcher_name']}: {s['K']:.1f} K, {s['BB']:.1f} BB, {s['H']:.1f} H, ERA={s['ERA_approx']}")

Predictions for yesterday (2026-04-09):

CIN @ MIA
  Rhett Lowder: 4.0 K, 2.3 BB, 5.7 H, ERA=4.49
  Max Meyer: 4.9 K, 1.8 BB, 5.2 H, ERA=3.82

ATH @ NYY
  Jeffrey Springs: 5.5 K, 1.6 BB, 4.8 H, ERA=4.19
  Ryan Weathers: 4.1 K, 1.6 BB, 4.9 H, ERA=4.12

DET @ MIN
  Jack Flaherty: 5.7 K, 1.5 BB, 4.1 H, ERA=3.47
  Mick Abel: 4.8 K, 1.7 BB, 4.3 H, ERA=4.31

AZ @ NYM
  Eduardo Rodriguez: 5.5 K, 2.1 BB, 5.9 H, ERA=3.99
  Nolan McLean: 5.8 K, 2.1 BB, 5.5 H, ERA=3.11

CWS @ KC
  Anthony Kay: 3.9 K, 1.9 BB, 4.8 H, ERA=4.73
  Seth Lugo: 4.8 K, 1.8 BB, 5.1 H, ERA=4.66

COL @ SD
  Jimmy Herget: 0.9 K, 0.3 BB, 1.0 H, ERA=3.46
  Randy Vásquez: 4.6 K, 1.9 BB, 5.6 H, ERA=3.77


## 6. Elite Pitcher Check

Verify predictions for elite K pitchers are improved.

In [8]:
# Find elite pitchers in recent predictions
all_preds = yesterday_preds + predictions

elite_k_threshold = 7.0  # Expected Ks
elite_pitchers = []

for game in all_preds:
    for pred_key in ["away_prediction", "home_prediction"]:
        pred = game.get(pred_key)
        if pred and pred["expected_stats"]["K"] >= elite_k_threshold:
            elite_pitchers.append({
                "name": pred["pitcher_name"],
                "K": pred["expected_stats"]["K"],
                "BB": pred["expected_stats"]["BB"],
                "H": pred["expected_stats"]["H"],
                "ERA": pred["expected_stats"]["ERA_approx"],
                "BF": pred["expected_bf"],
            })

if elite_pitchers:
    print(f"Elite K predictions (>= {elite_k_threshold} K):")
    elite_df = pd.DataFrame(elite_pitchers).sort_values("K", ascending=False)
    print(elite_df.to_string(index=False))
else:
    print("No elite K predictions found in recent games")

Elite K predictions (>= 7.0 K):
       name    K   BB    H  ERA   BF
Chase Burns 7.07 1.76 4.62 3.56 24.5


## 7. Summary Table

In [9]:
# Create summary DataFrame
rows = []

for game in predictions:
    for side, pred_key in [("away", "away_prediction"), ("home", "home_prediction")]:
        pred = game.get(pred_key)
        if pred:
            s = pred["expected_stats"]
            rows.append({
                "Game": f"{game['away_team']}@{game['home_team']}",
                "Pitcher": pred["pitcher_name"],
                "BF": pred["expected_bf"],
                "K": s["K"],
                "BB": s["BB"],
                "H": s["H"],
                "HR": s["HR"],
                "ER": s["ER"],
                "IP": s["IP_approx"],
                "ERA": s["ERA_approx"],
            })

if rows:
    summary_df = pd.DataFrame(rows)
    print(f"\nPrediction Summary for {today}:")
    print(summary_df.to_string(index=False))
else:
    print("No predictions available")


Prediction Summary for 2026-04-10:
   Game                 Pitcher   BF    K   BB    H   HR   ER  IP  ERA
PIT@CHC       Carmen Mlodzinski 21.0 3.95 1.90 4.55 0.52 2.20 4.3 4.57
PIT@CHC           Shota Imanaga 22.0 5.89 1.76 5.19 0.82 2.41 5.3 4.07
 AZ@PHI          Michael Soroka 19.5 4.13 1.66 4.42 0.58 1.90 4.3 3.94
 AZ@PHI           Jesús Luzardo 24.0 6.26 2.00 5.35 0.79 2.19 6.0 3.28
MIA@DET           Chris Paddack 21.5 3.93 1.84 4.43 0.64 2.08 4.3 4.33
MIA@DET          Keider Montero 20.0 3.76 1.99 5.23 0.70 2.46 4.7 4.75
LAA@CIN        Jack Kochanowicz 22.0 3.00 1.64 4.69 0.61 2.29 4.0 5.16
LAA@CIN             Chase Burns 24.5 7.07 1.76 4.62 0.82 2.11 5.3 3.56
MIN@TOR Simeon Woods Richardson 21.0 4.42 1.59 4.64 0.73 2.07 4.7 4.00
MIN@TOR          Patrick Corbin 20.0 3.68 1.67 4.43 0.56 1.97 4.3 4.09
ATH@NYM               J.T. Ginn 21.5 4.01 1.59 4.16 0.57 1.69 4.3 3.50
ATH@NYM             Clay Holmes 23.0 4.94 1.93 4.72 0.71 1.99 5.0 3.58
 NYY@TB                Luis Gil 23.0 4.62